# Native C++ Stimulus Execution

This notebook demonstrates the C++ hybrid backend stimulus path using an already-supported feature: Python precomputes stimulus coupling arrays by calling `stim.get_coupling(step)` for every simulation step, then the generated native loop adds those values after projection coupling and before afferent-coupling accumulation.

The example uses a single `Generic2dOscillator` subnetwork, a `StimuliRegion` pulse train applied to coupling variable 0, and compares three runs:

- Python hybrid simulator with stimulus
- Native C++ backend with the same stimulus
- Native C++ backend without stimulus as a control

## Environment Setup

The path setup mirrors the standalone example so the notebook can be run from the `backend_cpp/examples` directory or from `backend_cpp` with the notebook opened in Jupyter.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import tempfile
import warnings
from pathlib import Path

import numpy as np

cwd = Path.cwd().resolve()
if cwd.name == "examples":
    EXAMPLES_DIR = cwd
elif (cwd / "examples").is_dir() and (cwd / "backend.py").exists():
    EXAMPLES_DIR = cwd / "examples"
else:
    EXAMPLES_DIR = Path(".").resolve()

BACKEND_CPP_DIR = EXAMPLES_DIR.parent
SIMULATOR_DIR = BACKEND_CPP_DIR.parent
TVB_LIBRARY_ROOT = SIMULATOR_DIR.parent.parent
for path in (SIMULATOR_DIR, TVB_LIBRARY_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.environ.setdefault("TVB_USER_HOME", str(Path(tempfile.gettempdir()) / "tvb-user"))
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))
warnings.filterwarnings("ignore", message="Hybrid simulation is experimental: .*")

In [ ]:
import matplotlib.pyplot as plt

from tvb.datatypes import equations, patterns
from tvb.datatypes.connectivity import Connectivity
from tvb.simulator.backend_cpp import CppHybridBackend
from tvb.simulator.hybrid import NetworkSet, Simulator, Subnetwork
from tvb.simulator.integrators import HeunDeterministic
from tvb.simulator.models.oscillator import Generic2dOscillator
from tvb.simulator.monitors import TemporalAverage

## Configuration

`TAVG_PERIOD` controls the native backend `chunk_size`: one temporal-average sample is written for each `TAVG_PERIOD / DT` integration steps. The stimulus itself is evaluated at the integration step resolution.

In [ ]:
DT = 0.1
NNODES = 5
SIMULATION_LENGTH = 200.0
TAVG_PERIOD = 0.5
OUTPUT_DIR = EXAMPLES_DIR / "outputs"

## Stimulus Pattern

`StimuliRegion` needs a `Connectivity` object for its spatial component. Here we build a small 5-region connectivity only for stimulus weighting.

The pulse train starts at 10 ms, repeats every 30 ms, and lasts 8 ms per pulse. Node 0 receives the full spatial weight and node 1 receives half weight. Nodes 2-4 receive no direct stimulus.

In [ ]:
def make_stimulus_connectivity(n_nodes: int) -> Connectivity:
    conn = Connectivity(
        centres=np.ones((n_nodes, 3)),
        weights=np.ones((n_nodes, n_nodes), dtype=np.float64) * 0.1,
        tract_lengths=np.zeros((n_nodes, n_nodes), dtype=np.float64),
        region_labels=np.array([f"region_{i}" for i in range(n_nodes)]),
        speed=np.array([1.0]),
    )
    conn.configure()
    return conn


def make_pulse_stimulus(n_nodes: int) -> patterns.StimuliRegion:
    temporal = equations.PulseTrain()
    temporal.parameters["onset"] = 10.0
    temporal.parameters["T"] = 30.0
    temporal.parameters["tau"] = 8.0
    temporal.parameters["amp"] = 1.0

    weights = np.zeros(n_nodes, dtype=np.float64)
    weights[0] = 1.0
    weights[1] = 0.5

    return patterns.StimuliRegion(
        temporal=temporal,
        connectivity=make_stimulus_connectivity(n_nodes),
        weight=weights,
    )

In [ ]:
temporal_preview = make_pulse_stimulus(NNODES).temporal
t_eval = np.linspace(0.0, SIMULATION_LENGTH, 2000)
waveform = temporal_preview.evaluate(t_eval)

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.fill_between(t_eval, waveform, alpha=0.6, color="tab:orange")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Amplitude")
ax.set_title("PulseTrain temporal envelope")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Network Construction

`NetworkSet.add_stimulus()` wraps the `StimuliRegion` in a hybrid `Stim` object. The important arguments are:

- `target_name="g2d"`: selects the target subnetwork
- `stimulus_cvar=np.r_[0]`: injects into coupling variable slot 0
- `projection_scale=2.0`: multiplies the stimulus before it is added to coupling

The explicit `stim.configure(simulation_length)` call is needed before compiling with `CppHybridBackend`, because the native backend precomputes the full stimulus array during `compiled.run()`.

In [ ]:
def make_network(with_stimulus: bool, simulation_length: float) -> tuple[NetworkSet, Subnetwork]:
    model = Generic2dOscillator()
    model.configure()
    subnet = Subnetwork(
        name="g2d",
        model=model,
        scheme=HeunDeterministic(dt=DT),
        nnodes=NNODES,
    ).configure()
    subnet.node_indices = np.arange(NNODES)

    network = NetworkSet(subnets=[subnet], projections=[], stimuli=[])
    if with_stimulus:
        network.add_stimulus(
            target_name="g2d",
            stimulus=make_pulse_stimulus(NNODES),
            stimulus_cvar=np.r_[0],
            projection_scale=2.0,
        )

    network.configure()
    for stim in network.stimuli:
        stim.configure(simulation_length)
    return network, subnet


def make_initial_state(subnetwork: Subnetwork) -> np.ndarray:
    return np.zeros(
        (
            subnetwork.model.nvar,
            subnetwork.nnodes,
            subnetwork.model.number_of_modes,
        ),
        dtype=np.float64,
    )

## Python and Native Runners

The Python runner uses the normal hybrid `Simulator`. The native runner lowers the same `NetworkSet` with `CppHybridBackend`, builds a pybind11 extension, and calls `compiled.run()`.

Inside `compiled.run()`, `_build_stimulus_arrays()` creates a flat per-subnetwork array with logical shape `(n_cvar, n_nodes, nstep)`. The generated C++ loop reads the current step slice and adds it to the coupling buffer before integration.

In [ ]:
def run_python(
    network: NetworkSet,
    initial_state: np.ndarray,
    simulation_length: float,
    tavg_period: float,
) -> tuple[np.ndarray, np.ndarray]:
    sim = Simulator(
        nets=network,
        simulation_length=simulation_length,
        monitors=[TemporalAverage(period=tavg_period)],
    )
    sim.configure()
    ((times, data),) = sim.run(initial_conditions=[initial_state.copy()])
    return np.asarray(times, dtype=np.float64), np.asarray(data, dtype=np.float64)


def run_native(
    network: NetworkSet,
    initial_state: np.ndarray,
    simulation_length: float,
    tavg_period: float,
    source_hint: str,
) -> tuple[np.ndarray, np.ndarray, dict]:
    nstep = int(round(simulation_length / DT))
    chunk_size = int(round(tavg_period / DT))
    if chunk_size < 1:
        raise ValueError("tavg-period must be at least dt.")

    backend = CppHybridBackend(build_root=EXAMPLES_DIR / ".build")
    compiled = backend.compile(
        network,
        monitors=[TemporalAverage(period=tavg_period)],
        user_source_hint=source_hint,
    )
    ((times, data),) = compiled.run(
        nstep=nstep,
        chunk_size=chunk_size,
        initial_states=[initial_state.copy()],
    )
    return (
        np.asarray(times, dtype=np.float64),
        np.asarray(data, dtype=np.float64),
        compiled.debug_summary(),
    )


def max_abs_diff(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.max(np.abs(a - b)))

## Run the Comparisons

The Python and native stimulated runs should agree numerically. The native control run makes the stimulus effect visible without relying on the Python reference.

In [ ]:
stim_network_py, stim_subnet_py = make_network(
    with_stimulus=True,
    simulation_length=SIMULATION_LENGTH,
)
initial_state = make_initial_state(stim_subnet_py)
py_times, py_data = run_python(
    network=stim_network_py,
    initial_state=initial_state,
    simulation_length=SIMULATION_LENGTH,
    tavg_period=TAVG_PERIOD,
)

stim_network_cpp, stim_subnet_cpp = make_network(
    with_stimulus=True,
    simulation_length=SIMULATION_LENGTH,
)
cpp_times, cpp_data, cpp_summary = run_native(
    network=stim_network_cpp,
    initial_state=make_initial_state(stim_subnet_cpp),
    simulation_length=SIMULATION_LENGTH,
    tavg_period=TAVG_PERIOD,
    source_hint="native_stimulus_execution_notebook_pulsetrain",
)

control_network_cpp, control_subnet_cpp = make_network(
    with_stimulus=False,
    simulation_length=SIMULATION_LENGTH,
)
control_times, control_data, _ = run_native(
    network=control_network_cpp,
    initial_state=make_initial_state(control_subnet_cpp),
    simulation_length=SIMULATION_LENGTH,
    tavg_period=TAVG_PERIOD,
    source_hint="native_stimulus_execution_notebook_control",
)

assert py_data.shape == cpp_data.shape
assert control_data.shape == cpp_data.shape

## Check Numeric Agreement and Stimulus Effect

The native run should match the Python hybrid run for the same stimulated network. Node 0 should differ from the no-stimulus control. Node 4 has zero stimulus weight and no projections, so it should remain identical to control.

In [ ]:
summary = {
    "config": {
        "dt": DT,
        "nodes": NNODES,
        "simulation_length": SIMULATION_LENGTH,
        "tavg_period": TAVG_PERIOD,
        "stimulated_nodes": {"0": 1.0, "1": 0.5},
        "target_cvar": 0,
        "projection_scale": 2.0,
    },
    "shapes": {
        "python_data": list(py_data.shape),
        "native_data": list(cpp_data.shape),
        "control_data": list(control_data.shape),
    },
    "python_vs_native": {
        "time_max_abs": max_abs_diff(py_times, cpp_times),
        "data_max_abs": max_abs_diff(py_data, cpp_data),
        "data_rms": float(np.sqrt(np.mean((py_data - cpp_data) ** 2))),
    },
    "stimulus_effect_native": {
        "node0_stim_vs_control_max_abs": max_abs_diff(
            cpp_data[:, 0, 0, 0],
            control_data[:, 0, 0, 0],
        ),
        "node4_stim_vs_control_max_abs": max_abs_diff(
            cpp_data[:, 0, 4, 0],
            control_data[:, 0, 4, 0],
        ),
    },
    "native_debug": {
        "pipeline_stage": cpp_summary["pipeline_stage"],
        "stimulus_count": len(stim_network_cpp.stimuli),
        "generated_cpp_path": cpp_summary["generated_cpp_path"],
    },
}
print(json.dumps(summary, indent=2, sort_keys=True))

## Plot Python vs Native Stimulated Runs

This overlay plots the Python hybrid simulator and native C++ backend for the same stimulated network. The curves should lie on top of each other; the numeric summary above reports the exact maximum absolute difference.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(
    py_times,
    py_data[:, 0, 0, 0],
    color="tab:green",
    linewidth=2,
    label="Python hybrid",
)
axes[0].plot(
    cpp_times,
    cpp_data[:, 0, 0, 0],
    color="tab:red",
    linestyle=":",
    linewidth=2,
    label="Native C++",
)
axes[0].set_title("Stimulated node 0")
axes[0].set_ylabel("V")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(
    py_times,
    py_data[:, 0, 1, 0],
    color="tab:green",
    linewidth=2,
    label="Python hybrid",
)
axes[1].plot(
    cpp_times,
    cpp_data[:, 0, 1, 0],
    color="tab:red",
    linestyle=":",
    linewidth=2,
    label="Native C++",
)
axes[1].set_title("Partially stimulated node 1")
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("V")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.suptitle("Stimulated run: Python hybrid vs native C++")
fig.tight_layout()
plt.show()

## Plot Native Stimulated vs Control

This plot uses only native C++ output. The upper panel shows the directly stimulated node. The lower panel shows an unstimulated node, which remains aligned with the control because this example has no projections between nodes.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(
    cpp_times,
    cpp_data[:, 0, 0, 0],
    color="tab:red",
    label="Native C++ with PulseTrain",
)
axes[0].plot(
    control_times,
    control_data[:, 0, 0, 0],
    color="tab:gray",
    linestyle="--",
    label="Native C++ control",
)
axes[0].set_title("Node 0 receives full stimulus weight")
axes[0].set_ylabel("V")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(
    cpp_times,
    cpp_data[:, 0, 4, 0],
    color="tab:blue",
    label="Native C++ with PulseTrain",
)
axes[1].plot(
    control_times,
    control_data[:, 0, 4, 0],
    color="tab:gray",
    linestyle="--",
    label="Native C++ control",
)
axes[1].set_title("Node 4 has zero stimulus weight")
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("V")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.suptitle("Native C++ hybrid backend stimulus execution")
fig.tight_layout()
plt.show()

## Where the Native Stimulus Enters

For this backend feature, the Python wrapper prepares one contiguous stimulus array per subnetwork before entering the generated extension. The array stores all time steps and all target coupling slots. In the generated C++ simulation loop, the phase order is:

1. zero coupling buffers
2. accumulate intra-subnet projection coupling
3. accumulate inter-subnet projection coupling
4. add the precomputed stimulus contribution
5. accumulate afferent coupling monitor data
6. integrate subnet states

That order matches the Numba backend behavior: monitors that inspect afferent coupling see the final total input, including both projections and stimulus.